# 05 — Gate 2 on REAL indirect attacks (category-filtered)
Gate 2 on the full BIPIA set was contaminated: most 'attacks' were benign-ish instructions ('recommend a book', 'translate to Spanish'), so S=0.99 partly meant 'detector correctly scores benign-looking text as benign'. attack-minus-benign gap was ~0.

Here we keep only genuine injections and slice severity **by category**:
- **Harmful** (`BIPIA_HARMFUL`): Scams & Fraud, Misinformation & Propaganda -- a miss is unambiguously dangerous.
- **Hijack** (`BIPIA_HIJACK`): Base/cipher/reverse/emoji/translation -- model is made to deviate from its task (textbook injection).

**The decisive question:** do the detectors confidently miss the *harmful* category? If yes -> bulletproof headline. If they catch harmful ones and only miss mild -> the honest claim is 'over-reliance on surface cues', a different paper.

## Bootstrap

In [1]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
print('Session ready.')

Mounted at /content/drive
/content/drive/MyDrive/PICALIB_Research/picalib-research
Already up to date.
Session ready.


## Deps + HF login

In [2]:
!pip install -q transformers sentencepiece datasets
from huggingface_hub import login
import getpass
_t = getpass.getpass('HF token (blank to skip Prompt-Guard-2): ').strip()
if _t: login(token=_t); print('logged in')
else: print('no token - ProtectAI only')

HF token (blank to skip Prompt-Guard-2): ··········
logged in


## Load source + category-tagged malicious indirect

In [3]:
import importlib, data_loaders, detectors, metrics
for m in (data_loaders, detectors, metrics): importlib.reload(m)
from data_loaders import load_deepset, load_bipia_categorized, BIPIA_HARMFUL, BIPIA_HIJACK, BIPIA_MALICIOUS
from detectors import score_released, threshold_at_fpr, auroc
from metrics import fnr, severity_S
import numpy as np, pandas as pd, os

deepset = load_deepset()
bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS)
print('deepset', len(deepset), '| malicious indirect', len(bip))
print('categories:', sorted(bip[bip.label==1].meta.unique()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/500 [00:00<?, ?B/s]

data/train-00000-of-00001-9564e8b05b4757(…):   0%|          | 0.00/40.3k [00:00<?, ?B/s]

data/test-00000-of-00001-701d16158af8736(…):   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/546 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/116 [00:00<?, ? examples/s]

Cloning microsoft/BIPIA ...
BIPIA categorized: 210 attacks across 7 categories, 778 benigns
deepset 662 | malicious indirect 988
categories: ['Base Encoding', 'Emoji Substitution', 'Language Translation', 'Misinformation & Propaganda', 'Reverse Text', 'Scams & Fraud', 'Substitution Ciphers']


## Score both detectors (deepset cache reused; new malicious set scored once)

In [ ]:
DETECTORS = {'protectai_v2':'protectai/deberta-v3-base-prompt-injection-v2',
             'prompt_guard_2':'meta-llama/Llama-Prompt-Guard-2-86M'}
os.makedirs('data', exist_ok=True)
scores = {}
for tag, name in DETECTORS.items():
    fpd, fpb = f'data/score_{tag}_deepset.npy', f'data/score_{tag}_bipia_cat.npy'
    try:
        pds = np.load(fpd) if os.path.exists(fpd) else score_released(deepset.text.tolist(), name)
        if not os.path.exists(fpd): np.save(fpd, pds)
        if os.path.exists(fpb):
            print('load cached bipia_cat', tag); pbs = np.load(fpb)
        else:
            print('scoring malicious set', tag); pbs = score_released(bip.text.tolist(), name); np.save(fpb, pbs)
        scores[tag] = (pds, pbs); print(f'  {tag}: ok')
    except Exception as e:
        print(f'  [skip] {tag}: {e}')
print('scored:', list(scores.keys()))

scoring malicious set protectai_v2


config.json:   0%|          | 0.00/994 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  meta-llama/Llama-Prompt-Guard-2-86M: id2label={0: 'LABEL_0', 1: 'LABEL_1'} -> attack class index 1


## Per-category FNR + S (the real test)
Frozen threshold = 1% FPR on deepset benigns. For each attack category we report how often it's missed (FNR) and how confident those misses are (S).

In [ ]:
yd = deepset.label.values
rows = []
for tag,(pds,pbs) in scores.items():
    t = threshold_at_fpr(pds[yd==0], 0.01)
    benign_fpr = (pbs[bip.label.values==0] >= t).mean()
    for cat in sorted(bip[bip.label==1].meta.unique()):
        m = (bip.meta.values==cat)
        p = pbs[m]; y = np.ones(m.sum())
        miss = p < t
        S = float(np.mean(1-p[miss])) if miss.sum()>=1 else np.nan
        tier = 'HARMFUL' if cat in BIPIA_HARMFUL else 'hijack'
        rows.append({'detector':tag,'tier':tier,'category':cat,'n':int(m.sum()),
                     'FNR':round(float(miss.mean()),3),'n_misses':int(miss.sum()),
                     'S':(round(S,3) if not np.isnan(S) else 'NaN'),
                     'mean_p':round(float(p.mean()),3)})
    rows.append({'detector':tag,'tier':'(benign)','category':'benign_FPR',
                 'n':int((bip.label.values==0).sum()),'FNR':round(float(benign_fpr),3),
                 'n_misses':'-','S':'-','mean_p':round(float(pbs[bip.label.values==0].mean()),3)})
percat = pd.DataFrame(rows)
print(percat.to_string(index=False))

## Tier rollup: harmful vs hijack
The headline lives in the HARMFUL row: high FNR + high S there = detectors confidently wave through unambiguously-dangerous injections.

In [ ]:
tier_rows = []
for tag,(pds,pbs) in scores.items():
    t = threshold_at_fpr(pds[yd==0], 0.01)
    for tier,cats in [('HARMFUL',BIPIA_HARMFUL),('hijack',BIPIA_HIJACK)]:
        m = bip.meta.isin(cats).values & (bip.label.values==1)
        p = pbs[m]; miss = p < t
        S = float(np.mean(1-p[miss])) if miss.sum()>=1 else np.nan
        tier_rows.append({'detector':tag,'tier':tier,'n':int(m.sum()),
                          'FNR':round(float(miss.mean()),3),'n_misses':int(miss.sum()),
                          'S':(round(S,3) if not np.isnan(S) else 'NaN')})
tiers = pd.DataFrame(tier_rows)
print(tiers.to_string(index=False))

## Figure: harmful-attack score distribution (the money plot)

In [ ]:
import matplotlib.pyplot as plt
n=len(scores)
if n:
    fig,axes=plt.subplots(1,n,figsize=(6*n,4),squeeze=False)
    for ax,(tag,(pds,pbs)) in zip(axes[0],scores.items()):
        t = threshold_at_fpr(pds[yd==0],0.01)
        harm = pbs[bip.meta.isin(BIPIA_HARMFUL).values & (bip.label.values==1)]
        hij  = pbs[bip.meta.isin(BIPIA_HIJACK).values & (bip.label.values==1)]
        ax.hist(harm,bins=25,alpha=.7,density=True,label='HARMFUL attacks')
        ax.hist(hij,bins=25,alpha=.5,density=True,label='hijack attacks')
        ax.axvline(t,color='k',ls='--',lw=1,label=f't={t:.3f}')
        ax.set_title(tag); ax.set_xlabel('p=P(attack)'); ax.legend()
    plt.tight_layout(); os.makedirs('figures',exist_ok=True)
    plt.savefig('figures/gate2_malicious_by_tier.png',dpi=150); plt.show()
os.makedirs('reports',exist_ok=True)
percat.to_csv('reports/gate2_malicious_percat.csv',index=False)
tiers.to_csv('reports/gate2_malicious_tiers.csv',index=False)
print('saved figure + 2 CSVs')

## Verdict

In [ ]:
vlines=['HARMFUL-TIER READ (the decisive number):']
for _,r in tiers[tiers.tier=='HARMFUL'].iterrows():
    tag=r['detector']; vlines.append(f"[{tag}] harmful FNR={r['FNR']}, S={r['S']}, misses={r['n_misses']}/{r['n']}")
    if r['S']!='NaN' and float(r['S'])>0.5 and r['n_misses']>=10 and r['FNR']>0.3:
        vlines.append('   -> confidently misses UNAMBIGUOUSLY-HARMFUL injections. Bulletproof headline.')
    elif r['FNR']<0.2:
        vlines.append('   -> catches harmful injections. Headline shifts to surface-cue reliance (mild/hijack misses).')
    else:
        vlines.append('   -> mixed; inspect per-category table.')
verdict_text='\n'.join(vlines); print(verdict_text)

## Persist + commit

In [ ]:
from reslog import log_result
body = 'Category-filtered indirect (malicious only). Frozen deepset threshold @1% FPR.\n\nPER-CATEGORY:\n' + percat.to_string(index=False) + '\n\nTIERS:\n' + tiers.to_string(index=False) + '\n\n' + verdict_text
log_result('Gate 2 category-filtered (real indirect attacks)', body, csv_df=percat, csv_name='gate2_malicious_percat.csv')

In [ ]:
!git add -A
!git commit -m "Gate 2 category-filtered: per-category severity on genuine BIPIA injections (harmful vs hijack)"
!git push
print('Pushed.')